In [1]:
import pandas as pd
import scanpy as sc
from cellflow.model import CellFlow
import os
import pickle
import pandas as pd
import numpy as np
import scanpy as sc
import cellflow

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [2]:
data_dir = "/lustre/groups/ml01/workspace/ot_perturbation/models/cellflow/tahoe"
model = "charmed-deluge-4206_CellFlow.pkl"
wandb_name = model.split("_")[0]

In [3]:
df_cl_emb = pd.read_csv("/lustre/groups/ml01/workspace/ot_perturbation/data/sciplex_tahoe/cell_line_embedding_full_ccle_300_scaled.csv")
    

with open("/lustre/groups/ml01/workspace/ot_perturbation/data/sciplex_tahoe/id_to_cell_line.pkl", "rb") as f:
    id_to_cell_line = pickle.load(f)

ccle_embs_tahoe = {}
for cl in id_to_cell_line.keys():
    ccle_embs_tahoe[cl] = df_cl_emb[df_cl_emb["stripped_cell_line_name"]==id_to_cell_line[cl]][[str(el) for el in np.arange(300)]].values.squeeze()

ccle_embs_sciplex = {}
for cl in ["A549", "K562", "MCF7"]:
    ccle_embs_sciplex[cl] = df_cl_emb[df_cl_emb["stripped_cell_line_name"]==cl][[str(el) for el in np.arange(300)]].values.squeeze()


In [4]:
split = 5
adata_train_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/sciplex/adata_train_{split}.h5ad"
adata_ood_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/sciplex/adata_ood_{split}.h5ad"
adata_train_sci = sc.read_h5ad(adata_train_path)
adata_ood_sci = sc.read_h5ad(adata_ood_path)
adata_train_obs = adata_train_sci.obs.drop_duplicates(subset="condition")
adata_ood_obs = adata_ood_sci.obs.drop_duplicates(subset="condition")
sciplex_obs = pd.concat((adata_train_obs, adata_ood_obs))
sciplex_obs["dosage"] = sciplex_obs["logdose"]
sciplex_rep_dict = adata_train_sci.uns.copy()


In [5]:
cf = cellflow.model.CellFlow.load(os.path.join(data_dir, model))

In [6]:
df_cl_emb[df_cl_emb["stripped_cell_line_name"].isin(("A549", "MCF7", "K562"))]

,stripped_cell_line_name,0,1,2,3,4,5,6,7,8,...,290,291,292,293,294,295,296,297,298,299
145,MCF7,0.132416,0.829621,-1.022434,0.546540,0.325778,0.443667,-0.456900,-1.225514,0.027571,...,-0.347078,1.166020,1.346745,-0.722231,0.757360,1.599150,0.177833,2.412200,-0.573002,1.675591
297,A549,-0.596266,0.034091,0.301583,0.472836,0.713602,-2.127060,-0.418364,0.390761,0.866432,...,-0.479699,0.829538,-1.002900,0.443940,-0.921052,0.192163,0.049765,0.174340,0.124164,-0.097247
480,K562,1.160877,-0.158071,-0.300682,-0.441156,0.834041,-0.246539,-0.205357,-1.129235,-0.874907,...,1.056070,1.675273,0.480540,-0.067873,2.260952,2.325060,-0.795710,-0.211342,-1.208090,-0.162612


In [7]:
ccle_embs_sciplex = {}
for cl in ["A549", "K562", "MCF7"]:
    ccle_embs_sciplex[cl] = df_cl_emb[df_cl_emb["stripped_cell_line_name"]==cl][[str(el) for el in np.arange(300)]].values.squeeze()


In [8]:
out_dir = '/lustre/groups/ml01/workspace/ot_perturbation/models/cellflow/tahoe'

In [9]:
sciplex_obs["dosage"] = sciplex_obs["logdose"]
sciplex_rep_dict = adata_train_sci.uns.copy()
sciplex_rep_dict["cell_line_embeddings"] = ccle_embs_sciplex
sciplex_rep_dict["drug_embeddings"] = {k: np.concatenate((v, np.array([0]))) for k,v in sciplex_rep_dict["ecfp_dict"].items()}

In [10]:
sciplex_rep_dict["cell_line_embeddings"]["K562"][0], sciplex_rep_dict["cell_line_embeddings"]["K562"][1], sciplex_rep_dict["cell_line_embeddings"]["K562"][2]

(1.1608772452111489, -0.1580707511420354, -0.3006819868688421)

In [11]:
# embeddings directly from sciplex
df_sci_embedding, _ = cf.get_condition_embedding(sciplex_obs[sciplex_obs["drug"]!= "Vehicle"], condition_id_key="condition", rep_dict=sciplex_rep_dict)
df_sci_embedding["condition"] = df_sci_embedding.index
df_sci_embedding["cell_line"] = df_sci_embedding.apply(lambda x: x["condition"].split("_")[0], axis=1)
df_sci_embedding["dose"] = df_sci_embedding.apply(lambda x: x["condition"].split("_")[-1], axis=1)
df_sci_embedding["drug"] = df_sci_embedding.apply(lambda x: x["condition"].split("_")[1], axis=1)
df_sci_embedding.to_csv(os.path.join(out_dir, f"condition_embeddings_sciplex_{wandb_name}.csv"))




[########################################] | 100% Completed | 102.19 ms
[########################################] | 100% Completed | 203.72 ms
[########################################] | 100% Completed | 504.09 ms


In [12]:
tahoe_rep = cf.adata.uns.copy()
tahoe_rep["cell_line_embeddings"] = ccle_embs_tahoe

In [13]:
# cell line adapted to be the closest present one in TAHOE

ccle_embs_sciplex = {}
for cl in ["A549", "K562", "MCF7"]:
    ccle_embs_sciplex[cl] = df_cl_emb[df_cl_emb["stripped_cell_line_name"]==cl][[str(el) for el in np.arange(300)]].values.squeeze()

sciplex_rep_dict_updated = {}
sciplex_rep_dict_updated["drug_embeddings"] = {k: np.concatenate((v, np.array([0]))) for k,v in sciplex_rep_dict["ecfp_dict"].items()}
sciplex_rep_dict_updated["cell_line_embeddings"] = ccle_embs_sciplex


closest_cell_line = {}
for cell_line in ["K562", "MCF7", "A549"]:
    cell_line_emb = ccle_embs_sciplex[cell_line]
    closest_emb = None
    closest_dist = np.inf
    for cl, cl_emb in ccle_embs_tahoe.items():
        if np.mean((cell_line_emb - cl_emb)**2) < closest_dist:
            closest_dist = np.mean((cell_line_emb - cl_emb)**2)
            closest_emb = cl
    closest_cell_line[cell_line] = closest_emb

for k,v in closest_cell_line.items():
    sciplex_rep_dict_updated["cell_line_embeddings"][k] = ccle_embs_tahoe[v]

In [14]:
closest_cell_line

{'K562': 'CVCL_C466', 'MCF7': 'CVCL_C466', 'A549': 'CVCL_0023'}

In [15]:
sciplex_rep_dict_updated["cell_line_embeddings"]["K562"][0], sciplex_rep_dict_updated["cell_line_embeddings"]["K562"][1], sciplex_rep_dict_updated["cell_line_embeddings"]["K562"][2]

(-0.4050944636700771, -0.901323113304287, 1.1781502945000992)

In [16]:
df_cl_cl, _ = cf.get_condition_embedding(sciplex_obs[sciplex_obs["drug"]!= "Vehicle"], condition_id_key="condition", rep_dict=sciplex_rep_dict_updated)
df_cl_cl["condition"] = df_cl_cl.index
df_cl_cl["cell_line"] = df_cl_cl.apply(lambda x: x["condition"].split("_")[0], axis=1)
df_cl_cl["dose"] = df_cl_cl.apply(lambda x: x["condition"].split("_")[-1], axis=1)
df_cl_cl["drug"] = df_cl_cl.apply(lambda x: x["condition"].split("_")[1], axis=1)
df_cl_cl.to_csv(os.path.join(out_dir, f"condition_embeddings_sciplex_closest_cell_line_{wandb_name}.csv"))


[########################################] | 100% Completed | 102.17 ms
[########################################] | 100% Completed | 518.78 ms


In [17]:
sciplex_rep_dict = adata_train_sci.uns

In [18]:
sciplex_rep_dict = adata_train_sci.uns
sciplex_rep_dict["cell_line_embeddings"] = ccle_embs_sciplex
sciplex_rep_dict["drug_embeddings"] = {k: np.concatenate((v, np.array([0]))) for k,v in sciplex_rep_dict["ecfp_dict"].items()}

sciplex_rep_dict_updated = sciplex_rep_dict.copy()

closest_cell_line = {}
for cell_line in ["K562", "MCF7", "A549"]:
    cell_line_emb = ccle_embs_sciplex[cell_line]
    closest_emb = None
    closest_dist = np.inf
    for cl, cl_emb in ccle_embs_tahoe.items():
        if np.mean((cell_line_emb - cl_emb)**2) < closest_dist:
            closest_dist = np.mean((cell_line_emb - cl_emb)**2)
            closest_emb = cl
    closest_cell_line[cell_line] = closest_emb

for k,v in closest_cell_line.items():
    sciplex_rep_dict_updated["cell_line_embeddings"][k] = ccle_embs_tahoe[v]


In [19]:
# cell line from TAHOE, drug from sciplex, given one drug to be concatenated across cell lines
new_rows = []
constant_dose = 5.0

for drug in sciplex_obs["drug"].unique():
    for cl in cf.adata.obs["cell_line"].unique():
        new_rows.append([drug, cl, constant_dose])

new_obs2 = pd.DataFrame(new_rows, columns=["drug", "cell_line", "dosage"])
new_obs2["condition"] = new_obs2.apply(lambda x: f"{x['drug']}_{x['dosage']}_{x['cell_line']}", axis=1)
new_obs2["control"] = False


In [20]:
rep_dict_new = sciplex_rep_dict.copy()
rep_dict_new["cell_line_embeddings"] = tahoe_rep["cell_line_embeddings"]

In [21]:
df_mean3, _ = cf.get_condition_embedding(new_obs2[new_obs2["drug"]!= "Vehicle"], condition_id_key="condition", rep_dict=rep_dict_new)
df_mean3["condition"] = df_mean3.index
df_mean3["cell_line"] = df_mean3.apply(lambda x: x["condition"].split("_")[-1], axis=1)
df_mean3["dose"] = df_mean3.apply(lambda x: x["condition"].split("_")[1], axis=1)
df_mean3["drug"] = df_mean3.apply(lambda x: x["condition"].split("_")[0], axis=1)
df_mean3.to_csv(os.path.join(out_dir, f"sci_drug_with_tahoe_cl_{wandb_name}.csv"))


[########################################] | 100% Completed | 101.15 ms
[########################################] | 100% Completed | 102.10 ms
[########################################] | 100% Completed | 1.83 sms


In [22]:
len(np.unique(df_mean3["drug"]))

187

In [23]:
len(np.unique(df_mean3["cell_line"]))

45

In [24]:
df_mean3.shape

(8460, 132)

In [25]:
len(df_mean3["drug"].unique())

187

In [26]:
df_cl_cl.head()

,0,1,2,3,4,5,6,7,8,9,...,122,123,124,125,126,127,condition,cell_line,dose,drug
condition,,,,,,,,,,,,,,,,,,,,,
A549_(+)-JQ1_10.0,0.028278,0.052855,-0.011744,0.097530,-0.033990,0.012249,-0.015596,-0.037940,0.042625,0.097670,...,0.067650,-0.021730,0.113187,-0.027693,0.031387,0.066210,A549_(+)-JQ1_10.0,A549,10.0,(+)-JQ1
K562_(+)-JQ1_10.0,0.079705,0.100054,-0.003796,0.106389,-0.048031,-0.039795,0.043791,-0.035829,0.052077,0.134826,...,0.111272,0.056643,0.105749,0.014522,0.048716,0.140232,K562_(+)-JQ1_10.0,K562,10.0,(+)-JQ1
MCF7_(+)-JQ1_10.0,0.079705,0.100054,-0.003796,0.106389,-0.048031,-0.039795,0.043791,-0.035829,0.052077,0.134826,...,0.111272,0.056643,0.105749,0.014522,0.048716,0.140232,MCF7_(+)-JQ1_10.0,MCF7,10.0,(+)-JQ1
A549_(+)-JQ1_100.0,0.029472,0.051513,-0.012635,0.096617,-0.033332,0.012433,-0.018531,-0.037940,0.044055,0.096009,...,0.066471,-0.021705,0.113244,-0.025305,0.029367,0.063997,A549_(+)-JQ1_100.0,A549,100.0,(+)-JQ1
K562_(+)-JQ1_100.0,0.079864,0.098140,-0.003285,0.105745,-0.048718,-0.040243,0.039931,-0.035096,0.052086,0.132772,...,0.109510,0.055302,0.104941,0.016071,0.046957,0.136808,K562_(+)-JQ1_100.0,K562,100.0,(+)-JQ1


In [27]:
df_sci_embedding.head()

,0,1,2,3,4,5,6,7,8,9,...,122,123,124,125,126,127,condition,cell_line,dose,drug
condition,,,,,,,,,,,,,,,,,,,,,
A549_(+)-JQ1_10.0,0.028278,0.052855,-0.011744,0.097530,-0.033990,0.012249,-0.015596,-0.037940,0.042625,0.097670,...,0.067650,-0.021730,0.113187,-0.027693,0.031387,0.066210,A549_(+)-JQ1_10.0,A549,10.0,(+)-JQ1
K562_(+)-JQ1_10.0,0.031253,-0.106938,0.231260,-0.020106,-0.060555,-0.100118,-0.397807,0.137428,0.009357,0.035508,...,-0.291561,-0.104380,-0.062662,0.038194,-0.026199,-0.365561,K562_(+)-JQ1_10.0,K562,10.0,(+)-JQ1
MCF7_(+)-JQ1_10.0,-0.053707,-0.018642,0.014965,0.050347,0.057407,-0.018300,0.061184,-0.151043,0.026049,0.046175,...,-0.033518,-0.011263,-0.033532,-0.025019,0.062858,0.000279,MCF7_(+)-JQ1_10.0,MCF7,10.0,(+)-JQ1
A549_(+)-JQ1_100.0,0.029472,0.051513,-0.012635,0.096617,-0.033332,0.012433,-0.018531,-0.037940,0.044055,0.096009,...,0.066471,-0.021705,0.113244,-0.025305,0.029367,0.063997,A549_(+)-JQ1_100.0,A549,100.0,(+)-JQ1
K562_(+)-JQ1_100.0,0.040324,-0.115488,0.234013,-0.009866,-0.068812,-0.097497,-0.402001,0.141704,0.016793,0.035563,...,-0.288615,-0.112952,-0.051677,0.039173,-0.031351,-0.365175,K562_(+)-JQ1_100.0,K562,100.0,(+)-JQ1


In [28]:
df_mean3.head()

,0,1,2,3,4,5,6,7,8,9,...,122,123,124,125,126,127,condition,cell_line,dose,drug
condition,,,,,,,,,,,,,,,,,,,,,
(+)-JQ1_5.0_CVCL_0023,0.032130,0.046958,-0.015467,0.092884,-0.029986,0.014046,-0.026867,-0.037748,0.047401,0.090369,...,0.062342,-0.022234,0.112337,-0.018762,0.022974,0.056672,(+)-JQ1_5.0_CVCL_0023,0023,5.0,(+)-JQ1
(+)-JQ1_5.0_CVCL_0069,-0.003410,0.011010,0.012140,0.026062,0.010427,0.053547,0.044215,0.034263,-0.021387,0.036016,...,0.031318,-0.068020,0.023664,-0.059341,-0.023998,0.063091,(+)-JQ1_5.0_CVCL_0069,0069,5.0,(+)-JQ1
(+)-JQ1_5.0_CVCL_0131,0.052483,0.011475,0.101811,0.056888,-0.047217,-0.006834,0.015340,0.011638,0.058231,0.065937,...,-0.000207,-0.025129,0.000536,0.018715,-0.055982,0.032625,(+)-JQ1_5.0_CVCL_0131,0131,5.0,(+)-JQ1
(+)-JQ1_5.0_CVCL_0152,-0.043252,-0.001119,0.125949,0.070213,-0.052077,0.059476,-0.020126,0.154656,0.028072,-0.018730,...,0.088994,-0.151260,0.152292,-0.056447,-0.197629,-0.003739,(+)-JQ1_5.0_CVCL_0152,0152,5.0,(+)-JQ1
(+)-JQ1_5.0_CVCL_0179,0.158794,0.031812,0.129325,0.143312,-0.152035,-0.075834,-0.067905,0.033510,0.106927,0.163788,...,0.065931,-0.037435,0.133144,0.046644,-0.004921,0.048034,(+)-JQ1_5.0_CVCL_0179,0179,5.0,(+)-JQ1
